# Address Parser

**NLP IE — Split raw address text into structured fields**

## Project Overview

Parse US address strings into **street**, **city**, **state**, **zip** using regex.

## Learning Objectives

1. Parse semi-structured text with regex.
2. Handle varied address formats.
3. Validate extracted fields.

## Problem Statement

Extract street, city, state, zip from raw address text.

## Why This Project Matters

- Address parsing needed for geocoding and shipping.
- Inconsistent formats cause delivery failures.

## Dataset Overview

8 synthetic US addresses.

## Dataset Source and License Notes

Synthetic.

## Environment Setup

In [ ]:
import subprocess, sys, importlib
for pkg in ["pandas","numpy","matplotlib","seaborn"]:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
print("Environment ready.")


## Imports

In [ ]:
import re, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
%matplotlib inline
SEED = 42; np.random.seed(SEED)
print("Loaded.")


## Configuration / Constants

In [ ]:
US_STATES = {"CA":"California","NY":"New York","TX":"Texas","IL":"Illinois",
    "MA":"Massachusetts","WA":"Washington","DC":"Washington DC"}
print(f"State mappings: {len(US_STATES)}")


## Dataset Download or Loading

In [ ]:
ADDRESSES = [
    {"raw":"123 Main Street, Springfield, IL 62701","street":"123 Main Street","city":"Springfield","state":"IL","zip":"62701"},
    {"raw":"456 Oak Ave, Austin, TX 78701","street":"456 Oak Ave","city":"Austin","state":"TX","zip":"78701"},
    {"raw":"789 Broadway Apt 4B, New York, NY 10003","street":"789 Broadway Apt 4B","city":"New York","state":"NY","zip":"10003"},
    {"raw":"1000 Market St Suite 200, San Francisco, CA 94103","street":"1000 Market St Suite 200","city":"San Francisco","state":"CA","zip":"94103"},
    {"raw":"55 State St, Boston, MA 02109","street":"55 State St","city":"Boston","state":"MA","zip":"02109"},
    {"raw":"300 E Randolph St, Chicago, IL 60601","street":"300 E Randolph St","city":"Chicago","state":"IL","zip":"60601"},
    {"raw":"1 Infinite Loop, Cupertino, CA 95014","street":"1 Infinite Loop","city":"Cupertino","state":"CA","zip":"95014"},
    {"raw":"2200 Pennsylvania Ave NW, Washington, DC 20037","street":"2200 Pennsylvania Ave NW","city":"Washington","state":"DC","zip":"20037"},
]
df = pd.DataFrame(ADDRESSES)
print(f"Loaded {len(df)} addresses")


## Data Validation Checks

In [ ]:
assert df["raw"].notna().all()
print(f"Validated: {len(df)}")


## Exploratory Data Analysis

In [ ]:
df["chars"] = df["raw"].str.len()
print(f"Avg length: {df["chars"].mean():.0f}")


## Target Analysis

Targets: street, city, state, zip.

## Train/Validation/Test Split Strategy

Not applicable — rule-based.

## Preprocessing Strategy

Regex for `street, city, STATE zip`.

## Feature Engineering — Address Parser

In [ ]:
def parse_address(raw):
    m = re.match(r"^(.+),\s*([^,]+),\s*([A-Z]{2})\s+(\d{5}(?:-\d{4})?)$", raw.strip())
    if m: return {"street":m.group(1).strip(),"city":m.group(2).strip(),"state":m.group(3),"zip":m.group(4)}
    return {"street":None,"city":None,"state":None,"zip":None}
test = parse_address("123 Main St, Springfield, IL 62701")
print(f"Test: {test}")


## Baseline Model — Parse All Addresses

In [ ]:
parsed = df["raw"].apply(parse_address).apply(pd.Series)
for f in ["street","city","state","zip"]: df[f"ext_{f}"] = parsed[f]
print("Field accuracy:")
for f in ["street","city","state","zip"]:
    acc = (df[f] == df[f"ext_{f}"]).mean()
    print(f"  {f:8s}: {acc:.1%}")


## LazyPredict Benchmark

> **Not applicable.** NLP IE task.

## FLAML AutoML Run

> **Not applicable.** FLAML not used for NLP IE.

## Additional Best-Library Workflow — State Mapping

In [ ]:
df["state_name"] = df["ext_state"].map(US_STATES)
print(df[["ext_city","ext_state","state_name","ext_zip"]].to_string(index=False))


## Top 3 Model Selection

Single regex parser.

In [ ]:
fields = ["street","city","state","zip"]
accs = [(df[f] == df[f"ext_{f}"]).mean() for f in fields]
fig, ax = plt.subplots(figsize=(7,4))
ax.bar(fields, accs, color="steelblue")
ax.set_title("Address Extraction Accuracy"); ax.set_ylim(0,1.1)
plt.tight_layout(); plt.show()


## Final Training and Evaluation of Top 3

Regex parser is the final system.

## Error Analysis

In [ ]:
print("Checking for mismatches...")
errs = 0
for _, row in df.iterrows():
    for f in ["street","city","state","zip"]:
        if row[f] != row[f"ext_{f}"]: errs += 1; print(f"  Mismatch: {f}")
print(f"Total errors: {errs}")


## Interpretation and Business Insight

1. Regex works well for standard US addresses.
2. Apt/Suite numbers correctly included.
3. For production, use libpostal or USPS API.

## Limitations

- US only.
- Requires comma-separated format.
- No international support.

## How to Improve This Project

1. Use libpostal for multilingual.
2. Add USPS validation.
3. Support international formats.

## Production Considerations

- Use libpostal or Google Geocoding.
- Validate against USPS database.
- Cache results.

## Common Mistakes

1. Assuming consistent format.
2. Not handling multi-word cities.
3. Not validating zip vs state.

## Mini Challenge / Exercises

1. Add ZIP+4 support.
2. Parse addresses without zips.
3. Add geocoding.

## Final Summary / Key Takeaways

1. Regex works for consistent US addresses.
2. Real addresses vary — need robust parsers.
3. libpostal is the best open-source option.
4. Always validate before using.